In [86]:
import os
import streamlit as st
import time
from parser import *
from lxml import etree as ET
import pandas as pd


### Plik jest pobierany z aplikacji Zdrowie (Apple Health) na telefonie, format pliku XML,

### Najważniejsze typy danych: **Record|Correlation|Workout|ActivitySummary** - reszte czyli ClinicalRecord|Audiogram|VisionPrescription pomijamy.



In [87]:
path = "eksport.xml"

In [187]:
def discover_xml_structure(file_path):
    # Słownik: Klucz = Nazwa Tagu, Wartość = Zbiór atrybutów 'type' lub 'workoutActivityType'
    found_structures = {}

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        tag_name = elem.tag

        # Jeśli pierwszy raz widzimy ten tag, dodaj go do słownika
        if tag_name not in found_structures:
            found_structures[tag_name] = set()


        # Sprawdzamy, czy ten tag ma jakieś "podtypy" (np. HKQuantity...)
        sub_type = elem.get("type") or elem.get("workoutActivityType") or "Brak podtypu"

        found_structures[tag_name].add(sub_type)

        # Czyścimy RAM
        elem.clear()

    return found_structures

moje_dane = discover_xml_structure(path)

print("\n ODKRYTE ELEMENTY")
for tag, sub_types in moje_dane.items():
    print(f"\n TAG: <{tag}>")
    print(f"   Liczba różnych typów: {len(sub_types)}")
    for st in list(sub_types):
        print(f"     - {st}")

🚀 Skanuję plik w poszukiwaniu prawdy...

--- ODKRYTE ELEMENTY W TWOIM PLIKU ---

 TAG: <ExportDate>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Me>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Record>
   Liczba różnych typów: 41
     - HKQuantityTypeIdentifierWaistCircumference
     - HKQuantityTypeIdentifierVO2Max
     - HKQuantityTypeIdentifierSixMinuteWalkTestDistance
     - HKDataTypeSleepDurationGoal
     - HKQuantityTypeIdentifierFlightsClimbed
     - HKQuantityTypeIdentifierEnvironmentalAudioExposure
     - HKQuantityTypeIdentifierDistanceCycling
     - HKQuantityTypeIdentifierHeadphoneAudioExposure
     - HKQuantityTypeIdentifierStairDescentSpeed
     - HKQuantityTypeIdentifierHeartRateRecoveryOneMinute
     - HKQuantityTypeIdentifierBodyFatPercentage
     - HKQuantityTypeIdentifierTimeInDaylight
     - HKCategoryTypeIdentifierSleepAnalysis
     - HKCategoryTypeIdentifierAudioExposureEvent
     - HKQuantityTypeIdentifierDistanceWalkingRunning
     - HKQuant

In [190]:
important_tags = {
    'Me',"Record", "Workout", "WorkoutEvent", "WorkoutActivity",
    "WorkoutStatistics", "ActivitySummary", "InstantaneousBeatsPerMinute"
}


In [192]:

diff_types = set()

print("--- PRZEGLĄD STRUKTURY REKORDÓW ---")

# Iterujemy po pliku
for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag in important_tags:
        typ = elem.get("type") or elem.tag

        if typ not in diff_types:
            print(f"\n TYP: {typ}")
            print(f"   Atrybuty: {elem.attrib}")

            children = list(elem)
            if children:
                for child in children:
                    print(f"   └── Pod-element: <{child.tag}> atrybuty: {child.attrib}")

            diff_types.add(typ)



        elem.clear()

--- PRZEGLĄD STRUKTURY REKORDÓW ---

 TYP: Me
   Atrybuty: {'HKCharacteristicTypeIdentifierDateOfBirth': '2006-03-02', 'HKCharacteristicTypeIdentifierBiologicalSex': 'HKBiologicalSexFemale', 'HKCharacteristicTypeIdentifierBloodType': 'HKBloodTypeNotSet', 'HKCharacteristicTypeIdentifierFitzpatrickSkinType': 'HKFitzpatrickSkinTypeNotSet', 'HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse': 'Brak'}

 TYP: HKQuantityTypeIdentifierHeight
   Atrybuty: {'type': 'HKQuantityTypeIdentifierHeight', 'sourceName': 'FitnessOnline', 'sourceVersion': '206', 'unit': 'cm', 'creationDate': '2025-08-12 11:58:06 +0100', 'startDate': '2025-08-12 11:58:05 +0100', 'endDate': '2025-08-12 11:58:05 +0100', 'value': '163'}

 TYP: HKQuantityTypeIdentifierBodyMass
   Atrybuty: {'type': 'HKQuantityTypeIdentifierBodyMass', 'sourceName': 'Zdrowie', 'sourceVersion': '14.7.1', 'unit': 'kg', 'creationDate': '2021-09-06 10:46:43 +0100', 'startDate': '2021-09-06 10:46:00 +0100', 'endDate': '2021-09-06 10:46:00 +01

In [90]:
#Stworzenie dataframe z wybranym typem danych

short_names = {
    # --- CIAŁO / PODSTAWOWE ---
    'HKQuantityTypeIdentifierHeight': 'height',
    'HKQuantityTypeIdentifierBodyMass': 'weight',
    'HKQuantityTypeIdentifierWaistCircumference': 'waist',
    'HKQuantityTypeIdentifierBodyFatPercentage': 'fat_pct',

    # --- SERCE (HEART) ---
    'HKQuantityTypeIdentifierHeartRate': 'hr',
    'HKQuantityTypeIdentifierRestingHeartRate': 'rhr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage': 'hr_walk_avg',
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute': 'hrr_1min',
    'HKCategoryTypeIdentifierHighHeartRateEvent': 'hr_high_event',

    # --- AKTYWNOŚĆ / ENERGIA ---
    'HKQuantityTypeIdentifierStepCount': 'steps',
    'HKQuantityTypeIdentifierDistanceWalkingRunning': 'dist_walk',
    'HKQuantityTypeIdentifierDistanceCycling': 'dist_cycle',
    'HKQuantityTypeIdentifierBasalEnergyBurned': 'kcal_basal',
    'HKQuantityTypeIdentifierActiveEnergyBurned': 'kcal_active',
    'HKQuantityTypeIdentifierFlightsClimbed': 'flights',
    'HKQuantityTypeIdentifierAppleExerciseTime': 'exercise_min',
    'HKQuantityTypeIdentifierAppleStandTime': 'stand_min',
    'HKCategoryTypeIdentifierAppleStandHour': 'stand_hr',
    'HKQuantityTypeIdentifierPhysicalEffort': 'effort',
    'HKQuantityTypeIdentifierVO2Max': 'vo2max',

    # --- MOBILNOŚĆ / CHÓD ---
    'HKQuantityTypeIdentifierWalkingSpeed': 'walk_speed',
    'HKQuantityTypeIdentifierWalkingStepLength': 'walk_step_len',
    'HKQuantityTypeIdentifierWalkingAsymmetryPercentage': 'walk_asym',
    'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage': 'walk_dbl_supp',
    'HKQuantityTypeIdentifierStairAscentSpeed': 'stair_up_speed',
    'HKQuantityTypeIdentifierStairDescentSpeed': 'stair_down_speed',
    'HKQuantityTypeIdentifierAppleWalkingSteadiness': 'walk_steadiness',
    'HKQuantityTypeIdentifierSixMinuteWalkTestDistance': 'walk_6min',

    # --- ODDECH / POZIOMY ---
    'HKQuantityTypeIdentifierOxygenSaturation': 'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate': 'resp_rate',

    # --- SEN ---
    'HKCategoryTypeIdentifierSleepAnalysis': 'sleep',
    'HKDataTypeSleepDurationGoal': 'sleep_goal',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',

    # --- ŚRODOWISKO / DŹWIĘK ---
    'HKQuantityTypeIdentifierEnvironmentalAudioExposure': 'audio_env',
    'HKQuantityTypeIdentifierHeadphoneAudioExposure': 'audio_hp',
    'HKQuantityTypeIdentifierEnvironmentalSoundReduction': 'sound_red',
    'HKCategoryTypeIdentifierAudioExposureEvent': 'audio_event',
    'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent': 'audio_hp_event',
    'HKQuantityTypeIdentifierTimeInDaylight': 'daylight',

    # --- INNE ---
    'HKCategoryTypeIdentifierMenstrualFlow': 'menstr'
}


In [91]:


def load_all_health_data(file_path, mapping):
    """
    Funkcja zwraca słownik DataFrameów z krótkimi nazwami, żeby szybciej wpisywać.
    """
    if file_path is None:
        return "Zła ścieżka pliku, spróbuj ponownie"
    if mapping is None:
        mapping = types_list # jeśli nie mamy krótkich nazw używamy pobranych długich

    # 1. Tworzymy puste listy dla każdego krótkiego skrótu ze słownika
    data_buckets = {short_name: [] for short_name in mapping.values()}

    # 2. Startujemy iterparse
    # event="end" oznacza, że reagujemy, gdy zamknie się tag </Record>
    context = ET.iterparse(file_path, events=("end",))

    print("🚀 Rozpoczynam jednorazowy odczyt pliku... To potrwa ok. 30-60 sek.")

    for event, elem in context:
        if elem.tag == "Record":
            long_type = elem.get("type")

            # 3. Jeśli ten typ jest w naszym słowniku krótkich nazw
            if long_type in mapping:
                short_name = mapping[long_type]

                # Wyciągamy dane do słownika (lekki format)
                record = {
                    'value': elem.get('value'),
                    'unit': elem.get('unit'),
                    'sourceName': elem.get('sourceName'),
                    'startDate': elem.get('startDate'),
                    'endDate': elem.get('endDate'),

                }

                for child in elem:
                    if child.tag == "MetadataEntry":
                        record[child.get('key')] = child.get('value')

                data_buckets[short_name].append(record)

            elem.clear()



        root = context.root

    # 4. Zamieniamy listy na DataFrames i naprawiamy typy
    final_dfs = {}
    print("📊 Konwertuję dane na tabele...")

    for short_name, records in data_buckets.items():
        if records: # Tworzymy DF tylko jeśli są dane
            df = pd.DataFrame(records)

            # Automatyczna konwersja na liczby i daty
            if 'value' in df.columns:
                df['value'] = pd.to_numeric(df['value'], errors='coerce')
            if 'startDate' in df.columns:
                df['startDate'] = pd.to_datetime(df['startDate'])
            if 'endDate' in df.columns:
                df['endDate'] = pd.to_datetime(df['endDate'])

            final_dfs[short_name] = df

    print("✅ Wszystkie dane wczytane!")
    return final_dfs



hd = load_all_health_data("eksport.xml", short_names)


df = hd.get('hr')
df.head()
df.describe()
df_steps = hd.get('steps')

🚀 Rozpoczynam jednorazowy odczyt pliku... To potrwa ok. 30-60 sek.
📊 Konwertuję dane na tabele...
✅ Wszystkie dane wczytane!


In [92]:
print(df.head())
print(df.describe())
print(df_steps.head())

   value       unit sourceName                 startDate  \
0   72.0  count/min  Zepp Life 2021-12-17 00:00:00+01:00   
1   70.0  count/min  Zepp Life 2021-12-17 00:01:00+01:00   
2   70.0  count/min  Zepp Life 2021-12-17 00:02:00+01:00   
3   69.0  count/min  Zepp Life 2021-12-17 00:03:00+01:00   
4   70.0  count/min  Zepp Life 2021-12-17 00:04:00+01:00   

                    endDate HKMetadataKeyHeartRateMotionContext  \
0 2021-12-17 00:00:59+01:00                                 NaN   
1 2021-12-17 00:01:59+01:00                                 NaN   
2 2021-12-17 00:02:59+01:00                                 NaN   
3 2021-12-17 00:03:59+01:00                                 NaN   
4 2021-12-17 00:04:59+01:00                                 NaN   

  HKMetadataKeySyncVersion HKMetadataKeySyncIdentifier  
0                      NaN                         NaN  
1                      NaN                         NaN  
2                      NaN                         NaN  
3       

Czyścimy  dane - usuwamy kolumny w których wszystkie wartości sa nan

In [93]:
for name, df in hd.items():
    # axis=1 oznacza kolumny, how='all' tylko te całkiem puste
    hd[name] = df.dropna(axis=1, how='all')

In [94]:
for name in hd:
    hd[name].info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype                    
---  ------      --------------  -----                    
 0   value       1 non-null      int64                    
 1   unit        1 non-null      object                   
 2   sourceName  1 non-null      object                   
 3   startDate   1 non-null      datetime64[ns, UTC+01:00]
 4   endDate     1 non-null      datetime64[ns, UTC+01:00]
dtypes: datetime64[ns, UTC+01:00](2), int64(1), object(2)
memory usage: 172.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype                    
---  ------            --------------  -----                    
 0   value             7 non-null      float64                  
 1   unit              7 non-null      object                   
 2   sourceName        7 non-null      obj

In [125]:
hd['hr'].info()
print(hd['hr'].describe())
print(hd['hr'].tail(20))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 401424 entries, 0 to 401423
Data columns (total 6 columns):
 #   Column                               Non-Null Count   Dtype                    
---  ------                               --------------   -----                    
 0   value                                401424 non-null  float64                  
 1   unit                                 401424 non-null  object                   
 2   sourceName                           401424 non-null  object                   
 3   startDate                            401424 non-null  datetime64[ns, UTC+01:00]
 4   endDate                              401424 non-null  datetime64[ns, UTC+01:00]
 5   HKMetadataKeyHeartRateMotionContext  278951 non-null  object                   
dtypes: datetime64[ns, UTC+01:00](2), float64(1), object(3)
memory usage: 18.4+ MB
               value
count  401424.000000
mean       93.833004
std        28.793636
min        40.000000
25%        72.000000
5

In [167]:
df = hd['hr']


,value,unit,sourceName,startDate,endDate,HKMetadataKeyHeartRateMotionContext
0,72.0,count/min,Zepp Life,2021-12-17 00:00:00+01:00,2021-12-17 00:00:59+01:00,NaN
1,70.0,count/min,Zepp Life,2021-12-17 00:01:00+01:00,2021-12-17 00:01:59+01:00,NaN
2,70.0,count/min,Zepp Life,2021-12-17 00:02:00+01:00,2021-12-17 00:02:59+01:00,NaN
3,69.0,count/min,Zepp Life,2021-12-17 00:03:00+01:00,2021-12-17 00:03:59+01:00,NaN
4,70.0,count/min,Zepp Life,2021-12-17 00:04:00+01:00,2021-12-17 00:04:59+01:00,NaN
...,...,...,...,...,...,...
401419,107.0,count/min,Apple Watch (Lol),2026-03-05 14:39:22+01:00,2026-03-05 14:39:22+01:00,2
401420,107.0,count/min,Apple Watch (Lol),2026-03-05 14:39:27+01:00,2026-03-05 14:39:27+01:00,2
401421,101.0,count/min,Apple Watch (Lol),2026-03-05 14:39:32+01:00,2026-03-05 14:39:32+01:00,2
401422,107.0,count/min,Apple Watch (Lol),2026-03-05 14:39:56+01:00,2026-03-05 14:39:56+01:00,2


In [173]:
# Usuwamy konkretne kolumny techniczne, których nie potrzebujemy do analizy tętna
df = hd['hr'].drop(columns=['HKMetadataKeySyncVersion', 'HKMetadataKeySyncIdentifier'], errors='ignore')

In [169]:
print(df.columns)

Index(['value', 'unit', 'sourceName', 'startDate', 'endDate',
       'HKMetadataKeyHeartRateMotionContext'],
      dtype='object')


In [174]:



# Sprawdzamy, czy początek i koniec są takie same
# Wynikiem będzie True dla punktów (np. tętno) i False dla przedziałów (np. kroki)
df['is_point'] = df['startDate'] == df['endDate']

# Szybki podgląd: ile masz przedziałów, a ile punktów?
print(df['is_point'].value_counts())

is_point
True     278608
False       343
Name: count, dtype: int64


In [177]:
# Usuwamy rekordy, które mają IDENTYCZNY start, koniec I wartość
df_clean = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

print(f"Usunięto {len(df) - len(df_clean)} idealnych duplikatów.")

Usunięto 0 idealnych duplikatów.


In [178]:
df['midPoint'] = df['startDate'] + (df['endDate'] - df['startDate']) / 2

In [179]:
# Resampling dla kroków (sumujemy w oknie 1-minutowym)
df_steps_1min = df_steps.set_index('startDate')['value'].resample('1min').sum()

In [180]:
print(df.columns)

Index(['value', 'unit', 'sourceName', 'startDate', 'endDate',
       'HKMetadataKeyHeartRateMotionContext', 'is_point', 'midPoint'],
      dtype='object')


In [164]:
# 1. USUWANIE DUPLIKATÓW
# Czyścimy rekordy, które są identyczne (poczatek, koniec, wartość)
df = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

# Szukamy słowa "Apple" i ignorujemy to, co jest między wyrazami
df = df[df['sourceName'].str.contains('Apple', na=False)]


# 3. KONTROLA DATY (Inżynierskie BHP)
# Wywalamy rekordy, gdzie koniec jest przed początkiem (błędy systemowe)
df = df[df['endDate'] >= df['startDate']]

# 4. PRZYGOTOWANIE DO RESAMPLINGU
# Musimy ustawić datę jako indeks, żeby Pandas mógł "pogrupować" czas
df = df.set_index('startDate')

# 5. RESAMPLING (Ujednolicenie do 1 minuty)
# Zamiast 400k nieregularnych punktów, robimy średnią dla każdej minuty
df_hr_1min = df['value'].resample('1min').mean()

# 6. USUNIĘCIE "DZIUR" (Opcjonalnie)
# Jeśli przez godzinę nie miałaś zegarka, resampling stworzy tam NaN. Usuwamy je.
df_hr_1min = df_hr_1min.dropna()

print(f"Sukces! Twoje dane tętna są teraz jednolite. Zostało {len(df_hr_1min)} czystych minut.")

✅ Przefiltrowano dane po źródle.
Sukces! Twoje dane tętna są teraz jednolite. Zostało 168353 czystych minut.


In [165]:
display(df_hr_1min)

,value,unit,sourceName,endDate,HKMetadataKeyHeartRateMotionContext,duration,is_point,midPoint
startDate,,,,,,,,
2024-03-03 16:10:22+01:00,93.0,count/min,Apple Watch (Lol),2024-03-03 16:10:22+01:00,1,0 days,True,2024-03-03 16:10:22+01:00
2024-03-03 16:12:16+01:00,84.0,count/min,Apple Watch (Lol),2024-03-03 16:12:16+01:00,1,0 days,True,2024-03-03 16:12:16+01:00
2024-03-03 16:18:43+01:00,82.0,count/min,Apple Watch (Lol),2024-03-03 16:18:43+01:00,1,0 days,True,2024-03-03 16:18:43+01:00
2024-03-03 16:24:23+01:00,80.0,count/min,Apple Watch (Lol),2024-03-03 16:24:23+01:00,1,0 days,True,2024-03-03 16:24:23+01:00
2024-03-03 16:25:28+01:00,78.0,count/min,Apple Watch (Lol),2024-03-03 16:25:28+01:00,1,0 days,True,2024-03-03 16:25:28+01:00
...,...,...,...,...,...,...,...,...
2026-03-05 14:39:22+01:00,107.0,count/min,Apple Watch (Lol),2026-03-05 14:39:22+01:00,2,0 days,True,2026-03-05 14:39:22+01:00
2026-03-05 14:39:27+01:00,107.0,count/min,Apple Watch (Lol),2026-03-05 14:39:27+01:00,2,0 days,True,2026-03-05 14:39:27+01:00
2026-03-05 14:39:32+01:00,101.0,count/min,Apple Watch (Lol),2026-03-05 14:39:32+01:00,2,0 days,True,2026-03-05 14:39:32+01:00
